

<div style="display: flex; align-items: center;">
    <img src="https://github.com/zahern/data/raw/main/m.png" alt="My Image" style="width: 100px; margin-right: 20px;">
    <p><span style="font-size: 60px;"><strong>MetaCountRegressor</strong></span></p>
</div>

# Tutorial also available as a jupyter notebook
[Download Example Notebook](https://github.com/zahern/CountDataEstimation/blob/main/README.ipynb)

##### Quick Setup
The Below code demonstrates how to set up automatic optimization assisted by the harmony search algorithm. References to the Differential Evolution and Simulated Annealing has been mentioned (change accordingly)

## Quick install: Requires Python 3.10

Install `metacountregressor` using pip as follows:

```bash
pip install metacountregressor

In [1]:
!pip install metacountregressor --upgrade

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [2]:

import pandas as pd
import numpy as np
from metacountregressor.solution import ObjectiveFunction
from metacountregressor.metaheuristics import (harmony_search,
                                            differential_evolution,
                                            simulated_annealing)


loaded standard packages
loaded helper


#### Basic setup. 
The initial setup involves reading in the data and selecting an optimization algorithm. As the runtime progresses, new solutions will be continually evaluated. Finally, at the end of the runtime, the best solution will be identified and printed out. In the case of multiple objectives all of the best solutions will be printed out that belong to the Pareto frontier.

In [3]:
# Read data from CSV file
df = pd.read_csv(
"https://raw.githubusercontent.com/zahern/data/main/Ex-16-3.csv")
X = df
y = df['FREQ']  # Frequency of crashes
X['Offset'] = np.log(df['AADT']) # Explicitley define how to offset the data, no offset otherwise
# Drop Y, selected offset term and  ID as there are no panels
X = df.drop(columns=['FREQ', 'ID'])  

#some example argument, these are defualt so the following line is just for claritity. See the later agruments section for detials.
arguments = {'algorithm': 'hs', 'test_percentage': 0.15, 'test_complexity': 6, 'instance_number':1,
             'val_percentage':0.15, 'obj_1': 'bic', '_obj_2': 'RMSE_TEST', "_max_time": 6}
# Fit the model with metacountregressor
obj_fun = ObjectiveFunction(X, y, **arguments)
#replace with other metaheuristics if desired
results = harmony_search(obj_fun)



Setup Complete...
The type of models possible will consider: Poisson
--------------------------------------------------------------------------------
Log-Likelihood:  -15936.072791987432
--------------------------------------------------------------------------------
bic: 32067.66
--------------------------------------------------------------------------------
RMSE_TEST: 25.32
+-------------------------+---------+-------+----------+----------+------------+
|         Effect          | $\tau$  | Coeff | Std. Err | z-values | Prob |z|>Z |
+=========================+=========+=======+==========+==========+============+
| const                   | no      | -0.00 |   1.00   |  -0.00   | 1.00       |
+-------------------------+---------+-------+----------+----------+------------+
| WIDTH                   | arcsinh | -0.01 |   1.00   |  -0.01   | 0.99       |
+-------------------------+---------+-------+----------+----------+------------+
| MXMEDSH                 | arcsinh | 0.01  |   1.00 

# Preparing the Data
It is important to prepare the data as efficiently as possible. Here are some key points to consider:

 - The dataset should be formatted so that the tested variables are normalized appropriately.
 - Exclude variables that are not relevant if their purpose is unclear.
 - Ensure that encoding is handled properly for categorical data.
 - Efficient data preparation ensures better model performance and accuracy.

 A preparation stage has been given to you below to show how one should encode their data.

In [4]:
from metacountregressor import helperprocess

# First Drop Highly Correlated Features
# Keep terms we require, offset, frequency and any variables of interest.
df = helperprocess.interactions(df, keep = ['FREQ', 'Offset', 'LENGTH', 'AADT'])
print(df.columns)

# Configuration dictionary
config = {
   

    'AADT': {
        'type': 'continuous',
        'bounds': [0.0, np.inf],  # Filter values between these bounds
        'discrete': False,        # Specify if the data is continuous
        'apply_func': (lambda x: np.log(x + 1))  # Custom transformation (log)
    },
    'FREQ': {'type': 'none'}, #predictor we don't want to change this
    'LENGTH': {'type': 'none'},
    'Offset': {'type': 'none'}
}
#remove ID CoLUMNS from dataset
dataset = df.drop(columns = [
    'ID'
])
# for all remaining terms guess the type
for c in dataset.columns:
    if c not in config.keys():
        # FUNCTION Here to determine appropriate typings for other variavbles note deifined

        

        #if np.max(dataset[c])- np.min(dataset[c]) >20:
        #print(c)
        #print(np.quantile(dataset[c], .33))
        #print(helperprocess.guess_low_medium_high(c, dataset[c]))
        a = helperprocess.guess_low_medium_high(c, dataset[c])
        #config[c] = helperprocess.guess_column_type(c, dataset[c])
        config[c] = a
        
        
       
        


data_new  = helperprocess.transform_dataframe(dataset,config)
data_new.head()



Index(['ID', 'FREQ', 'LENGTH', 'INCLANES', 'SPEED', 'AADT', 'SINGLE', 'PEAKHR',
       'GRADEBR', 'MIGRADE', 'MXGRADE', 'MXGRDIFF', 'MINRAD', 'MEDWIDTH',
       'FRICTION', 'ADTLANE', 'SLOPE', 'INTECHAG', 'AVEPRE', 'AVESNOW',
       'Offset'],
      dtype='object')


,AADT,FREQ,LENGTH,Offset,INCLANES_Low,INCLANES_High,SPEED_Bin 1,SPEED_Bin 2,SINGLE_Low,SINGLE_Medium,...,SLOPE_High,INTECHAG_Low,INTECHAG_Medium,INTECHAG_High,AVEPRE_Low,AVEPRE_Medium,AVEPRE_High,AVESNOW_Low,AVESNOW_Medium,AVESNOW_High
0,8.116118,1,0.64,8.115820,False,True,False,True,False,False,...,True,False,True,False,False,False,True,False,False,False
1,8.116118,4,1.44,8.115820,False,True,False,True,False,False,...,True,False,True,False,False,False,True,False,False,False
2,9.441373,2,1.07,9.441293,False,True,False,True,False,False,...,False,False,True,False,False,True,False,False,False,True
3,9.441373,1,1.10,9.441293,False,True,False,True,False,False,...,False,False,True,False,False,True,False,False,False,True
4,9.543020,0,0.71,9.542948,False,True,False,True,False,False,...,False,False,True,False,True,False,False,False,False,True


# Preparing the data: Offset, Y, Grouping, Panels

When running models, certain terms—such as the offset term, panel column, and grouping column—are optional. However, specifying the Y value (the dependent variable) is mandatory. This means we must declare at least one column in our dataset to represent the Y value.

If you are planning to use grouped random parameters, you should define a grouping column. Similarly, if you are working with panel data, you need to specify the panel column. The same applies to the offset term—it must be declared if you intend to use it.

All of these terms should be organized into a dictionary, where the keys represent the term names (e.g., 'Y', 'grouped', 'panel', 'offset'), and the values correspond to the column names in your DataFrame. If you are not planning to use a specific term, simply set its value to None.

Below is an example of how to implement this in code:

In [ ]:
# Define the column mapping for model terms
model_terms = {
    'Y': 'FREQ',         # Replace 'FREQ' with the name of your dependent variable
    'group': 'county',    # Replace 'group_column' with the name of your grouping column (or None if not used)
    'panels': 'element_ID',      # Replace 'panel_column' with the name of your panel column (or None if not used)
    'Offset': 'Offset'                # Replace None with the name of your offset column if using one
}


# Preparing the constraints: Variable Constraints

'Metacountregressor' is a package that will search for the best possible model specifications. To reduce the burgeoning task of this process, an analyst should restrict the decision space as much as possible. In this case, follow the following code to control the freedom of each variable. That is by forcing specific variables to be some variables but not others, reducing the possible distributions for the random paramaters. The next code will step through this process. 

In [ ]:
variable_decisions = {
'Inclanes_Low': {'levels': [0,1,2,3,4,5], 'transformations': ['no'], 'distributions': ['u', 'ln', 'tn']},
'INCLANES_High': {'levels': [1], 'transformations': ['no'], 'distributions': []},
'SLOPE_High': {'levels': [0,1,2,3,4,5], 'transformations': ['no'], 'distributions': ['u', 'ln', 'tn']},
'SLOPE_Medium': {'levels': [1], 'transformations': ['no'], 'distributions': []},
'SLOPE_Low':{'levels': [1,3,6], 'transformations': ['no'], 'distributions': ['u', 'ln', 'tn']}
}
# this also renames X, so that the relavenet columns are named appropriately
a_des, X = helperprocess.set_up_analyst_constraints(data_new, model_terms, variable_decisions)


             Column  Level 0  Level 1  Level 2  Level 3  Level 4  Level 5  \
0              AADT     True     True     True     True     True     True   
1              FREQ     True     True     True     True     True     True   
2            LENGTH     True     True     True     True     True     True   
3            Offset     True     True     True     True     True     True   
4      INCLANES_Low     True     True     True     True     True     True   
5     INCLANES_High    False     True    False    False    False    False   
6       SPEED_Bin 1     True     True     True     True     True     True   
7       SPEED_Bin 2     True     True     True     True     True     True   
8        SINGLE_Low     True     True     True     True     True     True   
9     SINGLE_Medium     True     True     True     True     True     True   
10      SINGLE_High     True     True     True     True     True     True   
11       PEAKHR_Low     True     True     True     True     True     True   

## Arguments to feed into the Objective Function:
### 
Note: Please Consider the main arguments to change.

- `algorithm`: This parameter has multiple choices for the algorithm, such as 'hs', 'sa', and 'de'. Only one choice should be defined as a string value.
- `test_percentage`: This parameter represents the percentage of data used for in-sample prediction of the model. The value 0.15 corresponds to 15% of the data.
- `val_percentage`: This parameter represents the percentage of data used to validate the model. The value 0.15 corresponds to 15% of the data.
- `test_complexity`: This parameter defines the complexity level for testing. The value 6 tests all complexities. Alternatively, you can provide a list of numbers to consider different complexities. The complexities are further explained later in this document.
- `instance_number`: This parameter is used to give a name to the outputs.
- `_obj_1`: This parameter has multiple choices for obj_1, such as 'bic', 'aic', and 'hqic'. Only one choice should be defined as a string value.
- `_obj_2`: This parameter has multiple choices for objective 2, such as 'RMSE_TEST', 'MSE_TEST', and 'MAE_TEST'.
- `_max_time`: This parameter specifies the maximum number of seconds for the total estimation before stopping.
- `distribution`: This parameter is a list of distributions to consider. Please select all of the available options and put them into a list of valid options if you want to to consider the distribution type for use when modellign with random parameters. The valid options include: 'Normal', 'LnNormal', 'Triangular', and 'Uniform'.
- `transformations`: This parameters is a list of transformations to consider. Plesee select all of the available options and put them into a list of valid options if you want to consider the transformation type. The valid options include 'Normal', 'LnNormal', 'Triangular', 'Uniform'.
- `method_ll`: This is a specificication on the type of solvers are avilable to solve the lower level maximum likilihood objective. The valid options include: 'Normal', 'LnNormal', 'Triangular', and 'Uniform'.



### Example of changing the arguments:
Modify the arguments according to your preferences using the commented code as a guide.

In [ ]:
#Solution Arguments
arguments = {
        'algorithm': 'hs', #alternatively input 'de', or 'sa'
        'is_multi': 1,
        'test_percentage': 0.2, # used in multi-objective optimisation only. Saves 20% of data for testing.
        'val_percenetage:': 0.2, # Saves 20% of data for testing.
        'test_complexity': 6, # Complexity level for testing (6 tests all) or a list to consider potential differences in complexity
        'instance_number': 'name', # used for creeating a named folder where your models are saved into from the directory
        'distribution': ['Normal', 'LnNormal', 'Triangular', 'Uniform'],
        'Model': [0,1],  # or equivalently ['POS', 'NB']
        'transformations': ['no', 'sqrt', 'archsinh'],
        'method_ll': 'BFGS_2',
        '_max_time': 10, #time limit for algorthm 
        'decisions': a_des #anlyst decsions if defined, alternatively just comment out
    }
obj_fun = ObjectiveFunction(X, y, **arguments)
results = harmony_search(obj_fun)

Setup Complete...
The type of models possible will consider: Poisson
--------------------------------------------------------------------------------
Log-Likelihood:  -16704.856637136883
--------------------------------------------------------------------------------
bic: 33576.13
--------------------------------------------------------------------------------
MSE: 769.79
+-------------------------+---------+-------+----------+----------+------------+
|         Effect          | $\tau$  | Coeff | Std. Err | z-values | Prob |z|>Z |
+=========================+=========+=======+==========+==========+============+
| const                   | no      | -0.06 |   1.00   |  -0.06   | 0.95       |
+-------------------------+---------+-------+----------+----------+------------+
| LENGTH                  | log     | -0.07 |   1.00   |  -0.07   | 0.95       |
+-------------------------+---------+-------+----------+----------+------------+
| WIDTH                   | log     | -0.20 |   1.00   |  

## Initial Solution Configurement
Listed below is an example of how to specify an initial solution within the framework. This initial solution will be used to calculate the fitness and considered in the objective-based search. However, as the search progresses, different hypotheses may be proposed, and alternative modeling components may completely replace the initial solution.

In [6]:
 #Model Decisions, Specify for initial solution that will be optimised.
manual_fit_spec = {
    'fixed_terms': ['SINGLE', 'LENGTH'],
    'rdm_terms': ['AADT:normal'],
    'rdm_cor_terms': ['GRADEBR:normal', 'CURVES:normal'],
    'grouped_terms': [],
    'hetro_in_means': ['ACCESS:normal', 'MINRAD:normal'],
    'transformations': ['no', 'no', 'log', 'no', 'no', 'no', 'no'],
    'dispersion': 0
}


#Search Arguments
arguments = {
    'algorithm': 'hs',
    'test_percentage': 0.2,
    'test_complexity': 6,
    'instance_number': 'name',
    'Manual_Fit': manual_fit_spec
}
obj_fun = ObjectiveFunction(X, y, **arguments)

Setup Complete...
Benchmaking test with Seed 42
--------------------------------------------------------------------------------
Log-Likelihood:  -1339.1862434675106
--------------------------------------------------------------------------------
bic: 2732.31
--------------------------------------------------------------------------------
MSE: 650856.32
+--------------------------+--------+-------+----------+----------+------------+
|          Effect          | $\tau$ | Coeff | Std. Err | z-values | Prob |z|>Z |
+==========================+========+=======+==========+==========+============+
| LENGTH                   | no     | -0.15 |   0.01   |  -12.98  | 0.00***    |
+--------------------------+--------+-------+----------+----------+------------+
| SINGLE                   | no     | -2.46 |   0.04   |  -50.00  | 0.00***    |
+--------------------------+--------+-------+----------+----------+------------+
| GRADEBR                  | log    | 4.23  |   0.10   |  42.17   | 0.00***  

 Simarly to return the results feed the objective function into a metaheuristic solution algorithm. An example of this is provided below:

In [7]:
results = harmony_search(obj_fun)
print(results)

new solution, pareto optimal
--------------------------------------------------------------------------------
Log-Likelihood:  -756.1824091881238
--------------------------------------------------------------------------------
bic: 1666.89
--------------------------------------------------------------------------------
MSE: 3966.57
+-------------------------+---------+-------+----------+----------+------------+
|         Effect          | $\tau$  | Coeff | Std. Err | z-values | Prob |z|>Z |
+=========================+=========+=======+==========+==========+============+
| const                   | no      | -0.69 |   0.25   |  -2.75   | 0.01*      |
+-------------------------+---------+-------+----------+----------+------------+
| DECLANES                | no      | 0.15  |   0.20   |   0.72   | 0.47       |
+-------------------------+---------+-------+----------+----------+------------+
| WIDTH                   | arcsinh | -1.05 |   0.11   |  -9.71   | 0.00***    |
+-----------------

## Notes:
### Capabilities of the software include:
* Handling of Panel Data
* Support for Data Transformations
* Implementation of Models with Correlated and Non-Correlated Random Parameters
* A variety of mixing distributions for parameter estimations, including normal, lognormal, truncated normal, Lindley, Gamma, triangular, and uniform distributions
Capability to handle heterogeneity in the means of the random parameters
* Use of Halton draws for simulated maximum likelihood estimation
* Support for grouped random parameters with unbalanced groups
* Post-estimation tools for assessing goodness of fit, making predictions, and conducting out-of-sample validation
* Multiple parameter optimization routines, such as the BFGS method
* Comprehensive hypothesis testing using single objectives, such as in-sample BIC and log-likelihood
* Extensive hypothesis testing using multiple objectives, such as in-sample BIC and out-of-sample MAE (Mean Absolute Error), or in-sample AIC and out-of-sample MSPE (mean-square prediction errorr) 
* Features that allow analysts to pre-specify variables, interactions, and mixing distributions, among others
* Meta-heuristic Guided Optimization, including techniques like Simulated Annealing, Harmony Search, and Differential Evolution
* Customization of Hyper-parameters to solve problems tailored to your dataset
* Out-of-the-box optimization capability using default metaheuristics

### Intepreting the output of the model:
A regression table is produced. The following text elements are explained:
- Std. Dev.: This column appears for effects that are related to random paramters and displays the assument distributional assumption next to it
- Chol: This term refers to Cholesky decomposition element, to show the correlation between two random paramaters. The combination of the cholesky element on iyself is equivalent to a normal random parameter.
- hetro group: This term represents the heterogeneity group number, which refers all of the contributing factors that share hetrogentiy in the means to each other under the same numbered value.
- $\tau$: This column, displays the type of transformation that was applied to the specific contributing factor in the data.


## Arguments: 
#### In reference to the arguments that can be fed into the solution alrogithm, a dictionary system is utilised with relecant names these include


The following list describes the arguments available in this function. By default, all of the capabilities described are enabled unless specified otherwise as an argument. For list arguments, include all desired elements in the list to ensure the corresponding options are considered. Example code will be provided later in this guide.

1. **`complexity_level`**: This argument accepts an integer 1-6 or a list based of integegers between 0 to 5 eg might be a possible configuration [0, 2, 3]. Each integer represents a hierarchy level for estimable models associated with each explanatory variable. Here is a summary of the hierarchy:
    - 0: Null model
    - 1: Simple fixed effects model
    - 2: Random parameters model
    - 3: Random correlated parameters model
    - 4: Grouped random parameters model
    - 5: Heterogeneity in the means random parameter model

    **Note:** For the grouped random parameters model, groupings need to be defined prior to estimation. This can be achieved by including the following key-value pair in the arguments of the `ObjectiveFunction`: `'group': "Enter Column Grouping in data"`. Replace `"Enter Column Grouping in data"` with the actual column grouping in your dataset.

    Similarly, for panel data, the panel column needs to be defined using the key-value pair: `'panel': "enter column string covering panels"`. Replace `"enter column string covering panels"` with the appropriate column string that represents the panel information in your dataset.

2. **`distributions`**: This argument accepts a list of strings where each string corresponds to a distribution. Valid options include:
    - "Normal"
    - "Lindley"
    - "Uniform"
    - "LogNormal"
    - "Triangular"
    - "Gamma"
    - "TruncatedNormal"
    - Any of the above, concatenated with ":" (e.g., "Normal:grouped"; requires a grouping term defined in the model)

3. **`Model`**: This argument specifies the model form. It can be a list of integers representing different models to test:
    - 0: Poisson
    - 1: Negative-Binomial
    - 2: Generalized-Poisson

4. **`transformations`**: This argument accepts a list of strings representing available transformations within the framework. Valid options include:
    - "no"
    - "square-root"
    - "logarithmic"
    - "archsinh"
    - "as_factor"

5. **`is_multi`**: This argument accepts an integer indicating whether single or multiple objectives are to be tested (0 for single, 1 for multiple).

6. **`test_percentage`**: This argument is used for multi-objective optimization. Define it as a decimal; for example, 0.2 represents 20% of the data for testing.

7.  **`val_percentage`**: This argument saves data for validation. Define it as a decimal; for example, 0.2 represents 20% of the data for validation.

8. **`_max_time`**: This argument is used to add a termination time in the algorithm. It takes values as seconds. Note the time is only dependenant on the time after intial population of solutions are generated.

# Example: Assistance by Harmony Search


Let's begin by fitting very simple models and use the structure of these models to define our objectives. Then, we can conduct a more extensive search on the variables that are more frequently identified. For instance, in the case below, the complexity is level 3, indicating that we will consider, at most, three randomly correlated parameters. This approach is useful for initially identifying a suitable set of contributing factors for our search.


In [2]:
df = pd.read_csv(
"https://raw.githubusercontent.com/zahern/data/main/Ex-16-3.csv")
X = df
y = df['FREQ']  # Frequency of crashes
X['Offset'] = np.log(df['AADT']) # Explicitley define how to offset the data, no offset otherwise
# Drop Y, selected offset term and  ID as there are no panels
X = df.drop(columns=['FREQ', 'ID', 'AADT'])  

arguments = {
        'algorithm': 'hs', #alternatively input 'de', or 'sa'
        'is_multi': 1,
        'test_percentage': 0.2, # used in multi-objective optimisation only. Saves 20% of data for testing.
        'val_percentage:': 0.2, # Saves 20% of data for testing.
        'test_complexity': 3, # For Very simple Models
        'obj_1': 'BIC', '_obj_2': 'RMSE_TEST',
        'instance_number': 'name', # used for creeating a named folder where your models are saved into from the directory
        'distribution': ['Normal'],
        'Model': [0],  # or equivalently ['POS', 'NB']
        'transformations': ['no', 'sqrt', 'archsinh'],
        '_max_time': 10000
    }
obj_fun = ObjectiveFunction(X, y, **arguments)
results = harmony_search(obj_fun)
print(results)

NameError: name 'pd' is not defined

# Batched jobs for hyperparamater evaluation

Multiple experiments can be run, given enough resources such as a HPC. In this case we will show you how you can batch jobs to compare the potential solution algorithms for the search.

In [ ]:
# Harmony Search Hyperparameter Ranges
HS_PARAM_GRID = {
    '_hms': [10, 20, 30],  # Harmony Memory Size
    '_max_imp': [1000],  # Maximum Improvisations
    '_hmcr': [0.3, 0.5, 0.7],  # Harmony Memory Consideration Rate
    '_mpai': [1, 2, 3],  # Minimum Pitch Adjustment Index
}

# Differential Evolution Hyperparameter Ranges
DE_PARAM_GRID = {
    '_AI': [1, 2, 3],  # Adjustment Index
    '_crossover_perc': [0.2, 0.3, 0.4, 0.5, 0.6],  # Crossover Percentage
    '_max_iter': [1000],  # Maximum Iterations
    '_pop_size': [10, 25, 50],  # Population Size
}

# Simulated Annealing Hyperparameter Ranges
SA_PARAM_GRID = {
    'alpha': [0.9, 0.95, 0.99],  # Cooling Rate
    'STEPS_PER_TEMP': [5, 10, 20, 30],  # Steps Per Temperature
    'INTL_ACPT': [0.3, 0.5, 0.7],  # Initial Acceptance Rate
    '_crossover_perc': [0.2, 0.3, 0.4],  # Crossover Percentage
    'MAX_ITERATIONS': [1000],  # Fixed Value
    '_num_intl_slns': [10, 25, 50],  # Number of Initial Solutions
}


# Generate lists of parameter dictionaries for each algorithm
hs_combinations = helperprocess.generate_param_combinations(HS_PARAM_GRID)
de_combinations = helperprocess.generate_param_combinations(DE_PARAM_GRID)
sa_combinations = helperprocess.generate_param_combinations(SA_PARAM_GRID)

# Print example outputs
print("HS Combinations:", hs_combinations[:3]) # First 3 combinations for HS
print("DE Combinations:", de_combinations[:3]) # First 3 combinations for DE
print("SA Combinations:", sa_combinations[:3]) # First 3 combinations for SA

#batch harmony example
for args_hs in hs_combinations:
    obj_fun = ObjectiveFunction(X, y, **arguments)
    results = harmony_search(obj_fun, None, args_hs)
    print(results)



HS Combinations: [{'_hms': 10, '_max_imp': 1000, '_hmcr': 0.3, '_mpai': 1}, {'_hms': 10, '_max_imp': 1000, '_hmcr': 0.3, '_mpai': 2}, {'_hms': 10, '_max_imp': 1000, '_hmcr': 0.3, '_mpai': 3}]
DE Combinations: [{'_AI': 1, '_crossover_perc': 0.2, '_max_iter': 1000, '_pop_size': 10}, {'_AI': 1, '_crossover_perc': 0.2, '_max_iter': 1000, '_pop_size': 25}, {'_AI': 1, '_crossover_perc': 0.2, '_max_iter': 1000, '_pop_size': 50}]
SA Combinations: [{'alpha': 0.9, 'STEPS_PER_TEMP': 5, 'INTL_ACPT': 0.3, '_crossover_perc': 0.2, 'MAX_ITERATIONS': 1000, '_num_intl_slns': 10}, {'alpha': 0.9, 'STEPS_PER_TEMP': 5, 'INTL_ACPT': 0.3, '_crossover_perc': 0.2, 'MAX_ITERATIONS': 1000, '_num_intl_slns': 25}, {'alpha': 0.9, 'STEPS_PER_TEMP': 5, 'INTL_ACPT': 0.3, '_crossover_perc': 0.2, 'MAX_ITERATIONS': 1000, '_num_intl_slns': 50}]


See the table for parmaters that you can change 

## Paper

The following tutorial is in conjunction with our latest paper. A link the current paper is her [MetaCounteRegressor]()

## Contact
If you have any questions, ideas to improve MetaCountRegressor, or want to report a bug, just open a new issue in [GitHub repository](https://github.com/zahern/CountDataEstimation).

## Citing MetaCountRegressor
Please cite MetaCountRegressor as follows:

Ahern, Z., Corry P., Paz A. (2025). MetaCountRegressor [Computer software]. [https://pypi.org/project/metacounregressor/](https://pypi.org/project/metacounregressor/)

Or using BibTex as follows:

```bibtex
@misc{Ahern2024Meta,
   author = {Zeke Ahern, Paul Corry and Alexander Paz},
   journal = {PyPi},
   title = {metacountregressor · PyPI},
   url = {https://pypi.org/project/metacountregressor/0.1.80/},
   year = {2025},
}
